In [7]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import pyamg

# Example: 2D Poisson matrix (finite difference on a grid)
n = 200  # grid size -> matrix size ~ n^2
N = n * n
A = pyamg.gallery.poisson((n, n), format='csr')

# Right-hand side
b = np.random.rand(N)

# --- Baseline: CG without preconditioning
x0 = np.zeros_like(b)
%timeit x, info = spla.cg(A, b, x0=x0, atol=1e-8, maxiter=500)
# print("CG (no precond) info:", info)

# --- Build AMG preconditioner
ml = pyamg.ruge_stuben_solver(A)  # classical AMG hierarchy

# Wrap as LinearOperator for CG
M = ml.aspreconditioner()  

# --- CG with AMG preconditioning
%timeit x_prec, info_prec = spla.cg(A, b, x0=x0, atol=1e-8, maxiter=500, M=M)
# print("CG (with AMG) info:", info_prec)


316 ms ± 23.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
31.7 ms ± 779 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [14]:
x = np.random.random(2 * n * n)
y = np.random.random(n*n)

indexes_rand = np.random.choice(np.arange(2*n*n), size=n*n, replace=False)
indexes = np.arange(n * n)

# %timeit x[indexes] = y[indexes]
%timeit A @ x[indexes]
%timeit A @ x[indexes_rand]


203 μs ± 1.46 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
286 μs ± 3.92 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
